In [8]:
import pandas as pd
import matplotlib.pyplot as plt

# Helper Functions

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    # Since the Wind Degree is scrapped from the wind icon in the table, 
    # if the wind speed is too low, the icon will just be a circle without any rotation. 
    # Hence, they should be replaced by the corresponding degree of the Wind Direction.
    # Wind degree from timeanddate webpage
    wind_deg_dict = {
        'N': 90, 'NNE': 110, 'NE': 130, 'ENE': 150,
        'E': 180, 'ESE': 200, 'SE': 220, 'SSE': 240,
        'S': 270, 'SSW': 290, 'SW': 310, 'WSW': 330,
        'W': 360, 'WNW': 380, 'NW': 400, 'NNW': 430
    }

    df['Wind Degree'] = df['Wind Degree'].fillna(df['Wind Direction'].map(wind_deg_dict)) # Fill NA by Wind Direction
    
    # Standardize the Wind Degree
    df['Wind Degree'] = df['Wind Degree'] - 90
    
    # Adding time related columns
    df['Date'] = pd.to_datetime(df['Date'])
    df['Day'] = df['Date'].dt.day
    df['Month'] = df['Date'].dt.month
    df['Year'] = df['Date'].dt.year
    df['Week Number'] = df['Date'].dt.isocalendar().week
    
    # Add temperature difference
    df['Temp Diff'] = df['Temp High'] - df['Temp Low']
    
    return df

# Forecast Weather

In [10]:
forecast_df = pd.read_csv('csv/Los_Angeles_forecast_weather_data.csv')
forecast_df.head()

,Weekday,Date,Temp High,Temp Low,Condition,Feels Like,Humidity,Precipitation_Rain,Precipitation_Snow,Precipitation Chance,Wind Direction,Wind Degree,Wind Speed
0,Thu,May 14,74.0,57.0,Sunny,77.0,49.0,0.0,0.0,0.0,SSW,300.0,9.0
1,Fri,May 15,74.0,56.0,Mostly sunny,77.0,52.0,0.0,0.0,0.0,SSW,290.0,9.0
2,Sat,May 16,72.0,57.0,Scattered clouds,76.0,56.0,0.0,0.0,0.0,SSW,300.0,10.0
3,Sun,May 17,74.0,56.0,Mostly cloudy,77.0,47.0,0.0,0.0,0.0,S,280.0,10.0
4,Mon,May 18,78.0,57.0,Afternoon clouds,78.0,40.0,0.0,0.0,0.0,SW,320.0,10.0


In [11]:
forecast_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Weekday               15 non-null     object 
 1   Date                  15 non-null     object 
 2   Temp High             15 non-null     float64
 3   Temp Low              15 non-null     float64
 4   Condition             15 non-null     object 
 5   Feels Like            15 non-null     float64
 6   Humidity              15 non-null     float64
 7   Precipitation_Rain    15 non-null     float64
 8   Precipitation_Snow    15 non-null     float64
 9   Precipitation Chance  15 non-null     float64
 10  Wind Direction        15 non-null     object 
 11  Wind Degree           15 non-null     float64
 12  Wind Speed            15 non-null     float64
dtypes: float64(9), object(4)
memory usage: 1.7+ KB


## Weekday

In [12]:
forecast_df['Weekday'] = forecast_df['Weekday'].map({
    'Sun': 'Sunday',
    'Mon': 'Monday',
    'Tue': 'Tuesday',
    'Wed': 'Wednesday',
    'Thu': 'Thursday',
    'Fri': 'Friday',
    'Sat': 'Saturday'
})
forecast_df.head()

,Weekday,Date,Temp High,Temp Low,Condition,Feels Like,Humidity,Precipitation_Rain,Precipitation_Snow,Precipitation Chance,Wind Direction,Wind Degree,Wind Speed
0,Thursday,May 14,74.0,57.0,Sunny,77.0,49.0,0.0,0.0,0.0,SSW,300.0,9.0
1,Friday,May 15,74.0,56.0,Mostly sunny,77.0,52.0,0.0,0.0,0.0,SSW,290.0,9.0
2,Saturday,May 16,72.0,57.0,Scattered clouds,76.0,56.0,0.0,0.0,0.0,SSW,300.0,10.0
3,Sunday,May 17,74.0,56.0,Mostly cloudy,77.0,47.0,0.0,0.0,0.0,S,280.0,10.0
4,Monday,May 18,78.0,57.0,Afternoon clouds,78.0,40.0,0.0,0.0,0.0,SW,320.0,10.0


## Date

In [13]:
forecast_df['Date'] = forecast_df['Date'] + ', 2026'
forecast_df.head()

,Weekday,Date,Temp High,Temp Low,Condition,Feels Like,Humidity,Precipitation_Rain,Precipitation_Snow,Precipitation Chance,Wind Direction,Wind Degree,Wind Speed
0,Thursday,"May 14, 2026",74.0,57.0,Sunny,77.0,49.0,0.0,0.0,0.0,SSW,300.0,9.0
1,Friday,"May 15, 2026",74.0,56.0,Mostly sunny,77.0,52.0,0.0,0.0,0.0,SSW,290.0,9.0
2,Saturday,"May 16, 2026",72.0,57.0,Scattered clouds,76.0,56.0,0.0,0.0,0.0,SSW,300.0,10.0
3,Sunday,"May 17, 2026",74.0,56.0,Mostly cloudy,77.0,47.0,0.0,0.0,0.0,S,280.0,10.0
4,Monday,"May 18, 2026",78.0,57.0,Afternoon clouds,78.0,40.0,0.0,0.0,0.0,SW,320.0,10.0


## Clean Data

In [14]:
forecast_df = clean_data(forecast_df)
forecast_df.head()

,Weekday,Date,Temp High,Temp Low,Condition,Feels Like,Humidity,Precipitation_Rain,Precipitation_Snow,Precipitation Chance,Wind Direction,Wind Degree,Wind Speed,Day,Month,Year,Week Number,Temp Diff
0,Thursday,2026-05-14,74.0,57.0,Sunny,77.0,49.0,0.0,0.0,0.0,SSW,210.0,9.0,14,5,2026,20,17.0
1,Friday,2026-05-15,74.0,56.0,Mostly sunny,77.0,52.0,0.0,0.0,0.0,SSW,200.0,9.0,15,5,2026,20,18.0
2,Saturday,2026-05-16,72.0,57.0,Scattered clouds,76.0,56.0,0.0,0.0,0.0,SSW,210.0,10.0,16,5,2026,20,15.0
3,Sunday,2026-05-17,74.0,56.0,Mostly cloudy,77.0,47.0,0.0,0.0,0.0,S,190.0,10.0,17,5,2026,20,18.0
4,Monday,2026-05-18,78.0,57.0,Afternoon clouds,78.0,40.0,0.0,0.0,0.0,SW,230.0,10.0,18,5,2026,21,21.0


# Past Weather

In [15]:
past_df = pd.read_csv('csv/Los_Angeles_past_weather_data.csv')
past_df.head()

,Weekday,Date,Time,Temp High,Temp Low,Condition,Humidity,Barometer,Wind Direction,Wind Degree,Wind Speed
0,Wednesday,"April 29, 2026",12:00 pm — 6:00 pm,68.0,66.0,Sunny,59.0,29.96,W,350.0,12.428
1,Wednesday,"April 29, 2026",6:00 pm — 12:00 am,64.0,61.0,Clear,76.0,29.98,W,360.0,4.971
2,Thursday,"April 30, 2026",12:00 am — 6:00 am,61.0,61.0,Passing clouds,83.0,29.94,WSW,330.0,1.864
3,Thursday,"April 30, 2026",6:00 am — 12:00 pm,70.0,61.0,Overcast,66.0,29.91,ESE,210.0,4.350
4,Thursday,"April 30, 2026",12:00 pm — 6:00 pm,70.0,66.0,Sunny,61.0,29.86,W,350.0,11.807


In [16]:
past_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61 entries, 0 to 60
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Weekday         61 non-null     object 
 1   Date            61 non-null     object 
 2   Time            61 non-null     object 
 3   Temp High       61 non-null     float64
 4   Temp Low        61 non-null     float64
 5   Condition       61 non-null     object 
 6   Humidity        61 non-null     float64
 7   Barometer       61 non-null     float64
 8   Wind Direction  61 non-null     object 
 9   Wind Degree     57 non-null     float64
 10  Wind Speed      61 non-null     float64
dtypes: float64(6), object(5)
memory usage: 5.4+ KB


In [ ]:
past_df = clean_data(past_df)
past_df.head()

In [22]:
# Rename time of day
past_df['Time'] = past_df['Time'].map(
    {
        '12:00 am — 6:00 am': 1,
        '6:00 am — 12:00 pm': 2,
        '12:00 pm — 6:00 pm': 3,
        '6:00 pm — 12:00 am': 4
    }, 
)
past_df = past_df.rename(columns={'Time': 'Time of Day'})
past_df.head()

,Weekday,Date,Time of Day,Temp High,Temp Low,Condition,Humidity,Barometer,Wind Direction,Wind Degree,Wind Speed,Day,Month,Year,Week Number,Temp Diff
0,Wednesday,2026-04-29,3,68.0,66.0,Sunny,59.0,29.96,W,260.0,12.428,29,4,2026,18,2.0
1,Wednesday,2026-04-29,4,64.0,61.0,Clear,76.0,29.98,W,270.0,4.971,29,4,2026,18,3.0
2,Thursday,2026-04-30,1,61.0,61.0,Passing clouds,83.0,29.94,WSW,240.0,1.864,30,4,2026,18,0.0
3,Thursday,2026-04-30,2,70.0,61.0,Overcast,66.0,29.91,ESE,120.0,4.350,30,4,2026,18,9.0
4,Thursday,2026-04-30,3,70.0,66.0,Sunny,61.0,29.86,W,260.0,11.807,30,4,2026,18,4.0
